# 08 — ADAG (exact MAP) vs BP decoder gap

Compares the BP labellings against exact-MAP labellings produced by Schlesinger's ADAG solver (`manet/manet/adag_solver/`). The comparison runs on a random subsample of the visual Sudoku test split at three model checkpoints:

- `N=500, seed 0`, `N=1500, seed 0`, `N=5000, seed 0` — 100 random puzzles each, fixed `subsample_seed=42` so the same puzzle indices are used across all three.

## 0. Colab bootstrap (no-op locally)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import sys, os
    PROJECT_ROOT = '/content/drive/MyDrive/Master-thesis'
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT + '/notebooks')
    print('Colab setup done -- cwd:', os.getcwd())
except ImportError:
    print('Local run -- Colab bootstrap skipped')

## 1. Setup

In [ ]:
import json, os, sys, time, subprocess
from dataclasses import replace
from pathlib import Path

import numpy as np
import torch

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'mnlearn').is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('could not locate project root')
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

from mnlearn.config    import load_config
from mnlearn.data      import build_datasets
from mnlearn.models    import build_model
from mnlearn.inference import bp_decode

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS    = REPO_ROOT / 'results' / 'adag_vs_bp'
RESULTS.mkdir(parents=True, exist_ok=True)

BASE_CFG = load_config(REPO_ROOT / 'configs' / 'experiments' / 'lpm3n_visual_sudoku.yaml')

# Experiment design
CONFIGS = [
    {'N': 500,  'wd': 0.005, 'seed': 0},
    {'N': 1500, 'wd': 0.001, 'seed': 0},
    {'N': 5000, 'wd': 0.001, 'seed': 0},
]
N_PUZZLES      = 200
SUBSAMPLE_SEED = 42
TEST_SIZE      = 10000

print(f'DEVICE   = {DEVICE}')
print(f'configs  = {CONFIGS}')
print(f'puzzles  = {N_PUZZLES} per (N, seed); same subsample across all checkpoints')

## 2. Build the ADAG C++ library

One-time compile of `manet/manet/adag_solver/libadag.so` + CFFI Python module. Skipped if `libadag.so` is already present.

In [ ]:
import shutil

# 0. ensure cffi is available
try:
    import cffi  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cffi'], check=True)
    import cffi  # noqa

ADAG_DIR = REPO_ROOT / 'manet' / 'manet' / 'adag_solver'
LIB_PATH = ADAG_DIR / 'libadag.so'

# 1. compile libadag.so if missing (build in /tmp to avoid Drive I/O quirks)
TMP_DIR = Path('/tmp/adag_build')
TMP_DIR.mkdir(exist_ok=True)
shutil.copy(ADAG_DIR / 'libadag.cpp', TMP_DIR / 'libadag.cpp')
shutil.copy(ADAG_DIR / 'libadag.hpp', TMP_DIR / 'libadag.hpp')

if not LIB_PATH.exists():
    print('compiling libadag.so ...')
    r = subprocess.run(
        ['g++', '-O3', '-shared', '-std=c++11', '-fPIC',
         'libadag.cpp', '-o', 'libadag.so'],
        cwd=str(TMP_DIR), capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('STDOUT:', r.stdout); print('STDERR:', r.stderr)
        raise RuntimeError(f'g++ failed with exit {r.returncode}')
    shutil.copy(TMP_DIR / 'libadag.so', LIB_PATH)

    print('building CFFI wrapper ...')
    r = subprocess.run(
        [sys.executable, 'build_adag_cffi.py'],
        cwd=str(ADAG_DIR), capture_output=True, text=True,
    )
    if r.returncode != 0:
        print('STDOUT:', r.stdout); print('STDERR:', r.stderr)
        raise RuntimeError(f'CFFI build failed with exit {r.returncode}')
    print(f'  wrote {LIB_PATH}')
else:
    print(f'libadag.so already present at {LIB_PATH}, skipping build.')

# 2. The CFFI wrapper was built with rpath "." which resolves to CWD at import
# time. Since CWD is REPO_ROOT, copy libadag.so there so the loader finds it.
SHIM = REPO_ROOT / 'libadag.so'
if not SHIM.exists() or SHIM.stat().st_size != LIB_PATH.stat().st_size:
    shutil.copy(LIB_PATH, SHIM)
    print(f'copied libadag.so to CWD: {SHIM}')

# 3. add manet/ to sys.path and import
manet_root = str(REPO_ROOT / 'manet')
if manet_root not in sys.path:
    sys.path.insert(0, manet_root)
from manet.maxsum import adag, viterbi
print('manet.maxsum imported OK')

## 3. Load test split + Sudoku edges (shared across all N)

In [ ]:
test_X, test_Y = build_datasets(BASE_CFG.data, base_dir=REPO_ROOT)['test']
print(f'test_X: {test_X.shape}, test_Y: {test_Y.shape}')

# Reproducible random subsample
rng = np.random.RandomState(SUBSAMPLE_SEED)
SAMPLE_IDX = np.sort(rng.choice(test_X.shape[0], size=N_PUZZLES, replace=False))
print(f'subsample: {N_PUZZLES} indices from {test_X.shape[0]} (seed {SUBSAMPLE_SEED})')

# Build the Sudoku edge list once
_, EDGES_T = build_model(BASE_CFG.architecture, base_dir=REPO_ROOT)
EDGES_NUMPY = EDGES_T.cpu().numpy()   # [810, 2]
print(f'Sudoku edges: {EDGES_NUMPY.shape}')

# Convert to manet's [3, nE] format with edge-group 0 for all edges (single shared W)
def edges_to_manet(edges_np):
    """manet's adag expects E as [3, nE] with rows (node_i, node_j, pair_fn_idx).
    See manet/mn_models.py: self.E = np.concatenate((E, np.zeros((1, nE))), axis=0).
    Our model uses a single shared pairwise W so all edges get pair_fn_idx = 0."""
    nE = edges_np.shape[0]
    E = np.zeros((3, nE), dtype=np.int32)
    E[0, :] = edges_np[:, 0]  # node i
    E[1, :] = edges_np[:, 1]  # node j
    E[2, :] = 0               # pairwise function index (single shared W)
    return E

EDGES_MANET = edges_to_manet(EDGES_NUMPY)
print(f'manet edges: {EDGES_MANET.shape}')

## 4. Helpers — ADAG call + format conversion

In [ ]:
def fmt_dur(seconds):
    s = int(seconds)
    h = s // 3600; m = (s % 3600) // 60; ss = s % 60
    return f'{h:d}:{m:02d}:{ss:02d}' if h else f'{m:d}m{ss:02d}s'

def run_adag_one(unary_np, W_np, edges_manet):
    """Run ADAG on one puzzle.

    unary_np   : [81, K] float (per-cell logits after softmax — same scale as during training)
    W_np       : [K, K] float pairwise matrix
    edges_manet: [3, nE] int32, single edge-group

    Returns: labels [81] int, energy float
    """
    K, T = unary_np.shape[1], unary_np.shape[0]
    Q = unary_np.T.astype(np.float64)               # [K, T] = [9, 81] for adag
    G = np.expand_dims(W_np.astype(np.float64), 0)  # [1, K, K]
    return adag(Q, G, edges_manet)

## 5. Main loop — for each (N, seed), decode every sampled puzzle with both BP and ADAG

Prints `[i/N_PUZZLES] elapsed=... ETA=... avg=...s/puzzle` every 5 puzzles. Each (N, seed) result is saved to its own JSON in `results/adag_vs_bp/`.

In [ ]:
def evaluate_checkpoint(cfg_entry):
    N, wd, seed = cfg_entry['N'], cfg_entry['wd'], cfg_entry['seed']
    out_path = RESULTS / f'n{N}_seed{seed}_results.json'
    if out_path.exists():
        print(f'\n=== skip {out_path.name} (already exists) ===')
        return json.load(open(out_path))

    print(f'\n=== N={N}, seed={seed} ===')

    # Load model + checkpoint
    model, edges = build_model(BASE_CFG.architecture, base_dir=REPO_ROOT)
    run_dir = REPO_ROOT / 'results' / 'vis_sudoku_main' / f'run_n{N}_wd{wd:g}_seed{seed}'
    sd = torch.load(run_dir / 'model.pt', weights_only=True, map_location=DEVICE)
    model.load_state_dict(sd); model = model.to(DEVICE).eval()

    W_np = model.pairwise.detach().cpu().numpy()

    # Pre-compute all unaries in one batched forward (much faster than per-puzzle)
    with torch.no_grad():
        xb = test_X[SAMPLE_IDX].to(DEVICE)
        u  = model.unary(xb).cpu().numpy()    # [N_PUZZLES, 81, K]

    puzzle_results = []
    t_start = time.time()

    for i, idx in enumerate(SAMPLE_IDX):
        unary = u[i]
        y_gt  = test_Y[idx].cpu().numpy().astype(np.int64)

        # ADAG
        t0 = time.time()
        y_adag, e_adag = run_adag_one(unary, W_np, EDGES_MANET)
        t_adag = time.time() - t0

        with torch.no_grad():
            u_t = torch.from_numpy(unary).unsqueeze(0).to(DEVICE)
            y_bp = bp_decode(u_t, model.pairwise.to(DEVICE), edges.to(DEVICE),
                             num_iters=50)[0].cpu().numpy().astype(np.int64)

        y_adag = np.asarray(y_adag, dtype=np.int64)
        agree    = bool(np.array_equal(y_bp, y_adag))
        H_bp     = float(np.mean(y_bp   != y_gt))
        H_adag   = float(np.mean(y_adag != y_gt))
        zo_bp    = float(H_bp   > 0)
        zo_adag  = float(H_adag > 0)

        puzzle_results.append({
            'puzzle_idx': int(idx), 'agree': agree,
            'H_bp': H_bp,   'H_adag': H_adag,
            '01_bp': zo_bp, '01_adag': zo_adag,
            'energy_adag': float(e_adag),
            't_adag': t_adag,
        })

        if (i + 1) % 5 == 0 or (i + 1) == len(SAMPLE_IDX):
            elapsed = time.time() - t_start
            done    = i + 1
            avg     = elapsed / done
            eta     = avg * (len(SAMPLE_IDX) - done)
            agree_so_far = np.mean([r['agree'] for r in puzzle_results])
            print(f'  [{done:>3}/{len(SAMPLE_IDX)}]  elapsed={fmt_dur(elapsed):>8}  '
                  f'ETA={fmt_dur(eta):>8}  avg={avg:.1f}s/puzzle  '
                  f'BP-ADAG agree so far: {agree_so_far*100:.1f}%')

    out_path.write_text(json.dumps({'config': cfg_entry, 'puzzles': puzzle_results}, indent=2))
    print(f'  saved {out_path.name}')
    return {'config': cfg_entry, 'puzzles': puzzle_results}

all_results = [evaluate_checkpoint(cfg) for cfg in CONFIGS]

## 6. Summary table (printed + saved to results/adag_vs_bp/summary.json)

In [ ]:
def summarize(entry):
    cfg = entry['config']; puzzles = entry['puzzles']
    n = len(puzzles)
    agree     = np.array([p['agree']   for p in puzzles])
    H_bp      = np.array([p['H_bp']    for p in puzzles])
    H_adag    = np.array([p['H_adag']  for p in puzzles])
    zo_bp     = np.array([p['01_bp']   for p in puzzles])
    zo_adag   = np.array([p['01_adag'] for p in puzzles])
    return {
        'config':    cfg,
        'n_puzzles': n,
        'agreement_rate':    float(agree.mean()),
        'BP_01':             {'mean': float(zo_bp.mean()),   'std': float(zo_bp.std(ddof=1))},
        'ADAG_01':           {'mean': float(zo_adag.mean()), 'std': float(zo_adag.std(ddof=1))},
        'BP_hamming':        {'mean': float(H_bp.mean()),    'std': float(H_bp.std(ddof=1))},
        'ADAG_hamming':      {'mean': float(H_adag.mean()),  'std': float(H_adag.std(ddof=1))},
        'BP_strictly_worse_01_rate':   float(np.mean((zo_bp == 1) & (zo_adag == 0))),
        'ADAG_strictly_worse_01_rate': float(np.mean((zo_bp == 0) & (zo_adag == 1))),
    }

rows = [summarize(e) for e in all_results]

print(f'\n{"N":>6} {"agree":>7} {"BP 0/1":>14} {"ADAG 0/1":>14} {"BP H":>10} {"ADAG H":>10}  BP-only-loses')
print('-' * 90)
for r in rows:
    N = r['config']['N']
    print(f'{N:>6}  {r["agreement_rate"]*100:>5.1f}%  '
          f'{r["BP_01"]["mean"]:.3f}±{r["BP_01"]["std"]:.3f}    '
          f'{r["ADAG_01"]["mean"]:.3f}±{r["ADAG_01"]["std"]:.3f}    '
          f'{r["BP_hamming"]["mean"]:.4f}    {r["ADAG_hamming"]["mean"]:.4f}    '
          f'{r["BP_strictly_worse_01_rate"]*100:.1f}%')

(RESULTS / 'summary.json').write_text(json.dumps(rows, indent=2))
print(f'\nsaved {RESULTS / "summary.json"}')